# 01 — Audit, validation et nettoyage du dataset

**Projet :** Employee Attrition Prediction with Data Mining  
**Phase CRISP-DM :** 2. Data Understanding (et décisions de nettoyage)  
**Dataset :** `data/raw/employee_attrition_2026.csv`

## Objectifs

1. Charger le dataset et vérifier sa structure réelle.
2. Contrôler la qualité : valeurs manquantes, doublons, types, cohérence.
3. Analyser la variable cible `Attrition`.
4. Identifier les variables à exclure et les points à traiter lors du prétraitement.

> Ce notebook ne modifie pas le fichier source. Il documente l'état des données et prépare les décisions de l'étape de prétraitement.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
DATA_PATH = Path("..") / "data" / "raw" / "employee_attrition_2026.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("data") / "raw" / "employee_attrition_2026.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
print("Fichier :", DATA_PATH.resolve())

## 1. Chargement du dataset

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8")
print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head()

## 2. Structure et types de données

In [ ]:
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
})
structure

## 3. Qualité globale

In [ ]:
total_missing = int(df.isna().sum().sum())
duplicates = int(df.duplicated().sum())
print(f"Valeurs manquantes totales : {total_missing}")
print(f"Lignes dupliquées          : {duplicates}")
print(f"Colonnes dupliquées        : {int(df.columns.duplicated().sum())}")
print(f"Mémoire utilisée           : {df.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")

## 4. Variable cible `Attrition`

In [ ]:
target_counts = df["Attrition"].value_counts()
target_pct = df["Attrition"].value_counts(normalize=True).mul(100).round(2)
target_summary = pd.DataFrame({"effectif": target_counts, "pourcentage": target_pct})
target_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", color=["#2e7d32", "#c62828"], ax=ax)
ax.set_title("Distribution de la variable cible Attrition")
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'employés")
ax.bar_label(ax.containers[0], fmt="%d")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Lecture :** les classes sont presque équilibrées (52,45 % / 47,55 %). L'accuracy reste interprétable, mais le F1, le recall et le ROC-AUC seront privilégiés pour comparer les modèles.

## 5. Variables numériques

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
print(f"{len(numeric_cols)} colonnes numériques :")
print(numeric_cols)
df[numeric_cols].describe().T.round(2)

## 6. Variables catégorielles

In [ ]:
categorical_cols = [c for c in df.columns if c not in numeric_cols]
print(f"{len(categorical_cols)} colonnes catégorielles")
for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} modalités)")
    print(df[col].value_counts(dropna=False).to_string())

## 7. Identifiant `Employee ID`

In [ ]:
id_col = "Employee ID"
print(f"Valeurs uniques : {df[id_col].nunique():,} / {len(df):,}")
print(f"Plage           : {df[id_col].min()} - {df[id_col].max()}")
print(f"Est unique      : {df[id_col].is_unique}")

`Employee ID` est un identifiant unique sans pouvoir prédictif. Il sera **exclu des variables explicatives** pour éviter toute fuite d'information.

## 8. Contrôles de cohérence

In [ ]:
checks = {
    "Age hors de [18, 65]": int((~df["Age"].between(18, 65)).sum()),
    "Monthly Income <= 0": int((df["Monthly Income"] <= 0).sum()),
    "Years at Company > Age - 16": int((df["Years at Company"] > df["Age"] - 16).sum()),
    "Company Tenure > Age": int((df["Company Tenure"] > df["Age"]).sum()),
    "Number of Dependents < 0": int((df["Number of Dependents"] < 0).sum()),
}
for label, count in checks.items():
    print(f"{label:38s} : {count:,} ligne(s)")

**Interprétation :**

- `Company Tenure` dépasse `Age` pour **42 283 lignes (70,94 %)**. Cela n'est cohérent que si `Company Tenure` est exprimé **en mois** et `Age` **en années** : l'unité doit être clarifiée avant utilisation.
- `Years at Company` dépasse `Age - 16` pour **19 050 lignes (31,96 %)**, ce qui impliquerait un recrutement avant 16 ans. C'est une incohérence connue du dataset synthétique ; elle sera documentée et éventuellement traitée au prétraitement.
- Aucune valeur d'âge, de revenu ou de personnes à charge n'est aberrante.

## 9. Décisions de nettoyage et de prétraitement

In [ ]:
print("Modalités de Education Level :")
for value in sorted(df["Education Level"].unique()):
    print(f"  - {value!r}")

Décisions retenues pour l'étape 2 (prétraitement) :

1. **Exclure `Employee ID`** des variables explicatives.
2. **Aucune imputation** : zéro valeur manquante.
3. **Aucune suppression de doublon** : zéro doublon (contrôle conservé par sécurité).
4. **Normaliser l'apostrophe** de `Education Level` (`’` → `'`).
5. **Encoder les variables ordinales** dans leur ordre naturel.
6. **Encoder en one-hot** les variables nominales.
7. **Encoder en binaire** les variables `Yes` / `No`.
8. **Standardiser** les variables numériques.
9. **Documenter l'unité de `Company Tenure`** (probablement en mois).
10. **Fit du prétraitement sur le train uniquement** via `Pipeline` / `ColumnTransformer`.

## 10. Synthèse de l'audit

In [ ]:
audit_summary = pd.DataFrame([
    {"contrôle": "Lignes", "valeur": f"{df.shape[0]:,}"},
    {"contrôle": "Colonnes", "valeur": df.shape[1]},
    {"contrôle": "Variables explicatives", "valeur": df.shape[1] - 2},
    {"contrôle": "Valeurs manquantes", "valeur": total_missing},
    {"contrôle": "Doublons", "valeur": duplicates},
    {"contrôle": "Stayed", "valeur": f"{target_counts['Stayed']:,} ({target_pct['Stayed']} %)"},
    {"contrôle": "Left", "valeur": f"{target_counts['Left']:,} ({target_pct['Left']} %)"},
])
audit_summary

## Conclusion

Le dataset est **exploitable en l'état** : 59 598 lignes, 24 colonnes, aucune valeur manquante, aucun doublon, et une cible binaire presque équilibrée. Les points de vigilance sont l'identifiant à exclure, l'unité de `Company Tenure` et deux incohérences temporelles propres au dataset synthétique.

L'étape suivante consiste à implémenter le prétraitement dans `notebooks/03_preprocessing.ipynb` et `src/preprocessing.py`, puis les tests associés.